# Feature Engineering

**Family:** Fundamentals | **Level:** Beginner | **Runs on:** CPU, under a minute

Scaling, encoding, interactions and a leakage safe pipeline.

## Learning objectives

- Scale and encode features inside a pipeline
- Create interaction and transformed features
- Compare feature sets with cross validation
- Avoid target leakage in encodings

In [1]:
import warnings
warnings.filterwarnings("ignore")
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display
from ml_lab.data import load_dataset
from ml_lab.utils import set_seed
from ml_lab import visualization as viz
SEED = 42
set_seed(SEED)
viz.apply_style()

## 1. A pipeline for mixed columns

Telco churn has numeric and categorical columns and missing values. The preprocessor is fitted on training rows only.

In [2]:
from ml_lab.preprocessing import split_three_way, build_preprocessor
bundle = load_dataset("telco_churn")
X, y = bundle.X, bundle.y
Xtr, Xva, Xte, ytr, yva, yte = split_three_way(X, y, task="classification", seed=SEED)
prep = build_preprocessor(Xtr)
Z = prep.fit_transform(Xtr)
print("raw columns:", X.shape[1], "| after encoding:", Z.shape[1])
print("missing values after imputation:", int(np.isnan(Z).sum()))

raw columns: 19 | after encoding: 45
missing values after imputation: 0


## 2. Transformations

A skewed feature often works better after a log transform.

In [3]:
from scipy.stats import skew
df = load_dataset("auto_mpg").frame
print("skew of displacement:", round(float(skew(df["displacement"])), 2), "| after log:", round(float(skew(np.log(df["displacement"]))), 2))

skew of displacement: 0.7 | after log: 0.21


## 3. Does an engineered feature help?

We compare cross validated RMSE with and without `weight / horsepower` and `log(weight)`.

In [4]:
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import cross_val_score, KFold
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import StandardScaler
base = df[["weight", "horsepower", "model_year", "cylinders"]]
eng = base.assign(power_to_weight=df["horsepower"] / df["weight"], log_weight=np.log(df["weight"]))
for name, X_ in [("base", base), ("engineered", eng)]:
    s = cross_val_score(make_pipeline(StandardScaler(), LinearRegression()), X_, df["mpg"], cv=KFold(5, shuffle=True, random_state=SEED), scoring="neg_root_mean_squared_error")
    print(f"{name}: CV RMSE {-s.mean():.3f} +- {s.std(ddof=1):.3f}")

base: CV RMSE 3.433 +- 0.440
engineered: CV RMSE 3.005 +- 0.315


## 4. Target leakage in encoding

Encoding a category with the mean of the target computed on all rows leaks the label. Inside cross validation use the training fold only. Here we use a high cardinality random column to make the effect visible.

In [5]:
rng = np.random.default_rng(SEED)
ids = rng.integers(0, 150, len(df))                       # random category, no signal
y = df["mpg"].values
leak_enc = pd.Series(y).groupby(ids).transform("mean").values
from sklearn.model_selection import cross_val_predict
pred = cross_val_predict(LinearRegression(), leak_enc.reshape(-1, 1), y, cv=5)
print("R2 of a random category with leaky target encoding:", round(1 - ((y - pred) ** 2).sum() / ((y - y.mean()) ** 2).sum(), 3), "(the true signal is zero)")

R2 of a random category with leaky target encoding: 0.182 (the true signal is zero)


## Exercises

1. Add `model_year` as a categorical column and compare with the numeric version.
2. Implement out-of-fold target encoding for `ids` and show that R2 falls to about zero.

## Summary

- Fit transformers on training rows only.
- Engineered features need cross validated proof.
- Target encoding must be out of fold.

**Next:** Supervised learning: linear regression.

**Data note.** Real datasets come from `ml_lab.data` with licence notes in `datasets/catalog`. Synthetic numbers appear only where a demonstration needs controlled data, and are labelled.